# City friends — do they start their day later in the chat? (week 4, distributions)

Builds the Analysis 14 chart from `analysis-log.md`.

**Question (Stage 1):** city friends start work later, so on weekdays their first message
of the day comes later than their non-city friends'.

**Test, fixed before looking:** per person, the median time of their first message on
weekdays (day starts 06:00; only days they sent ≥ 5 messages; ≥ 20 such days to count).
Yes if at least 4 of the 5 city people are later than all 4 non-city people.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from wa_analyzer.data import load_own_chat

own = load_own_chat()
# same fix as 06-09: `timestamp` is stored as strings; the `+00:00` is only a label --
# these are the raw export's local clock times.
own["timestamp"] = pd.to_datetime(own["timestamp"], utc=True).dt.tz_localize(None)

DAY_STARTS_HOUR = 6
MIN_MESSAGES_PER_DAY = 5
MIN_DAYS = 20
PERIOD = (pd.Timestamp("2023-08-01"), pd.Timestamp("2026-10-01"))
CITY = {"striking-rail", "pliable-tiger", "animated-elk", "humorous-stingray", "effervescent-penguin"}

## First message of the day

A message before 06:00 belongs to the previous day, so a late night never counts as an
early start. Weekdays only, because the reasoning is about when work starts.

In [ ]:
shifted = own["timestamp"] - pd.Timedelta(hours=DAY_STARTS_HOUR)
own["day"] = shifted.dt.normalize()
own["minutes_after_6"] = (shifted - own["day"]).dt.total_seconds() / 60

in_period = own[(own["day"] >= PERIOD[0]) & (own["day"] < PERIOD[1]) & (own["day"].dt.weekday < 5)]
days = in_period.groupby(["author", "day"]).agg(
    first_message=("minutes_after_6", "min"), messages_that_day=("minutes_after_6", "size")
).reset_index()
days["first_message_hour"] = DAY_STARTS_HOUR + days["first_message"] / 60
days["is_city"] = days["author"].isin(CITY)
qualifying = days[days["messages_that_day"] >= MIN_MESSAGES_PER_DAY]

print(f"{len(days)} person-weekdays with any message, {len(qualifying)} with >= {MIN_MESSAGES_PER_DAY}")

## Stage 3 — shape (groups not shown yet)

How many qualifying days each person has, and what first-message times look like for
everyone pooled — not split by city/non-city, so the result isn't seen before the method
is settled.

In [ ]:
days_per_person = qualifying.groupby("author").size().sort_values()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].barh(days_per_person.index, days_per_person, color="grey")
axes[0].axvline(MIN_DAYS, color="black", ls="--", lw=1)
axes[0].set_xlabel(f"weekdays with ≥ {MIN_MESSAGES_PER_DAY} messages")
axes[0].set_title(f"Qualifying days per person (line = {MIN_DAYS})")
axes[1].hist(qualifying["first_message_hour"], bins=[h / 2 for h in range(12, 61)], color="grey")
axes[1].set_xlabel("time of first message (hour; day starts 06:00)")
axes[1].set_ylabel("person-days")
axes[1].set_title("First message of the day, everyone pooled")
plt.tight_layout()
plt.show()

print(days_per_person.to_string())
qualifying["first_message_hour"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(2)

## The test (Stage 3 decisions)

Daytime days only (first message before 18:00 — an evening first message is a different kind
of day). A person needs ≥ 20 such days. That leaves 3 city vs 3 non-city. Yes if all 3 city
people have a later median than all 3 non-city people (5% by chance).

In [ ]:
DAYTIME_ENDS_HOUR = 18

daytime = qualifying[qualifying["first_message_hour"] < DAYTIME_ENDS_HOUR]
people = daytime.groupby("author").agg(
    typical_start=("first_message_hour", "median"), days=("first_message_hour", "size"), is_city=("is_city", "first")
)
people = people[people["days"] >= MIN_DAYS].sort_values("typical_start")
people["group"] = people["is_city"].map({True: "city", False: "non-city"})

city_starts = people.loc[people["is_city"], "typical_start"]
non_city_starts = people.loc[~people["is_city"], "typical_start"]
print(f"all {len(city_starts)} city later than all {len(non_city_starts)} non-city: "
      f"{city_starts.min() > non_city_starts.max()}")
people.assign(typical_start_clock=pd.to_datetime(people["typical_start"], unit="h").dt.strftime("%H:%M"))

## Stage 4 — sketches (scratch)

A = distributions (share of daytime days per hour, per group) · B = categories (the test:
one dot per person) · C = relations (the stretch: start time vs how much someone posts).

In [ ]:
GROUP_COLORS = {"city": "#d95f02", "non-city": "#999999"}
tested = daytime[daytime["author"].isin(people.index)]
hours = range(6, DAYTIME_ENDS_HOUR)

fig, ax = plt.subplots(figsize=(9, 4))
width = 0.4
for offset, (group, colour) in zip([-width / 2, width / 2], GROUP_COLORS.items()):
    in_group = tested[tested["is_city"] == (group == "city")]
    per_person = in_group.groupby("author")["first_message_hour"].apply(
        lambda h: h.astype(int).value_counts(normalize=True).reindex(hours, fill_value=0)
    ).unstack().mean()  # each person one vote
    ax.bar([h + offset for h in hours], per_person, width=width, color=colour, label=f"{group} (3 people)")
ax.set_xticks(list(hours), [f"{h:02d}" for h in hours])
ax.set_xlabel("first message of a weekday (hour)")
ax.set_ylabel("share of days")
ax.set_title("Sketch A — when the first message comes, city vs non-city")
ax.legend(frameon=False)
fig.savefig("sketch-A-city-first-message-hours.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.8))
for group, colour in GROUP_COLORS.items():
    rows = people[people["group"] == group]
    ax.scatter(rows["typical_start"], [0] * len(rows), color=colour, s=120, label=group, zorder=3)
    for name, row in rows.iterrows():
        ax.annotate(name, (row["typical_start"], 0), xytext=(0, 12), textcoords="offset points",
                    rotation=45, fontsize=8, color=colour)
ax.set_yticks([])
ax.set_xlabel("median time of the first message (hour)")
ax.set_title("Sketch B — one dot per person: are all city dots right of all non-city dots?")
ax.legend(frameon=False, loc="lower right")
fig.savefig("sketch-B-city-first-message-people.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
volume = days[days["author"].isin(people.index)].groupby("author")["messages_that_day"].mean()
fig, ax = plt.subplots(figsize=(6, 4.5))
for group, colour in GROUP_COLORS.items():
    rows = people[people["group"] == group]
    ax.scatter(volume[rows.index], rows["typical_start"], color=colour, s=100, label=group)
    for name in rows.index:
        ax.annotate(name, (volume[name], rows.loc[name, "typical_start"]), xytext=(5, 3),
                    textcoords="offset points", fontsize=8)
ax.set_xlabel("average messages per active weekday")
ax.set_ylabel("median first-message time (hour)")
ax.set_title("Sketch C — is a late start just a quiet person?")
ax.legend(frameon=False)
fig.savefig("sketch-C-city-first-message-vs-volume.png", dpi=120, bbox_inches="tight")
plt.show()

## Chart after the Stage 5 critique

Sketch A with a stronger colour for the city group and a shorter axis label. Title is a
placeholder for the defensible claim; wording is the student's.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for offset, (group, colour) in zip([-width / 2, width / 2], GROUP_COLORS.items()):
    in_group = tested[tested["is_city"] == (group == "city")]
    per_person = in_group.groupby("author")["first_message_hour"].apply(
        lambda h: h.astype(int).value_counts(normalize=True).reindex(hours, fill_value=0)
    ).unstack().mean()
    ax.bar([h + offset for h in hours], per_person, width=width, color=colour, label=f"{group} (3 people)")
ax.set_xticks(list(hours), [f"{h:02d}" for h in hours])
ax.set_xlabel("first message of a weekday (hour)")
ax.set_ylabel("share of days")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("City friends don't start their day later in the chat", loc="left")
ax.legend(frameon=False)
fig.savefig("city-first-message-v2.png", dpi=120, bbox_inches="tight")
plt.show()

## Stage 6 — held-out check: the movers *before* their move

Rule fixed before running: if the movers were already earlier than the non-city people
before they moved, being early is a personal trait, not the city.

In [ ]:
NON_CITY = ["vibrant-barracuda", "fluffy-beaver", "rib-tickling-curlew"]
MOVES = {"animated-elk": "2022-09-01", "humorous-stingray": "2023-07-01", "effervescent-penguin": "2023-07-01"}


def typical_starts(start, end, authors, min_days=MIN_DAYS, daytime_ends=DAYTIME_ENDS_HOUR):
    """Median daytime first-message hour per person, same rules as the main test."""
    window = own[(own["day"] >= start) & (own["day"] < end) & (own["day"].dt.weekday < 5) & own["author"].isin(authors)]
    per_day = window.groupby(["author", "day"]).agg(first=("minutes_after_6", "min"), n=("minutes_after_6", "size"))
    per_day["hour"] = DAY_STARTS_HOUR + per_day["first"] / 60
    kept = per_day[(per_day["n"] >= MIN_MESSAGES_PER_DAY) & (per_day["hour"] < daytime_ends)]
    summary = kept.groupby("author")["hour"].agg(typical_start="median", days="size")
    return summary[summary["days"] >= min_days]


EXPORT_START = own["day"].min()
before = pd.concat({
    mover: typical_starts(EXPORT_START, pd.Timestamp(moved), [mover, *NON_CITY])
    for mover, moved in MOVES.items()
}, names=["window_of"])
before["clock"] = pd.to_datetime(before["typical_start"], unit="h").dt.strftime("%H:%M")
before

## Stage 6 — sensitivity (secondary, the primary test stays as fixed)

In [ ]:
checks = {
    "primary (20 days, before 18:00)": typical_starts(*PERIOD, list(CITY) + NON_CITY + ["hypnotic-rabbit"]),
    "15 days": typical_starts(*PERIOD, list(CITY) + NON_CITY + ["hypnotic-rabbit"], min_days=15),
    "real morning (before 12:00)": typical_starts(*PERIOD, list(CITY) + NON_CITY + ["hypnotic-rabbit"], daytime_ends=12),
}
for name, result in checks.items():
    order = result.sort_values("typical_start").assign(group=lambda r: r.index.map(lambda a: "city" if a in CITY else "non-city"))
    print(f"{name}: " + ", ".join(f"{a} ({g}, {pd.to_datetime(h, unit='h'):%H:%M}, {d}d)"
                                  for a, g, h, d in zip(order.index, order["group"], order["typical_start"], order["days"])))